# S&P 500 Market Direction Prediction

This notebook builds a machine-learning pipeline to predict whether the S&P 500 will close higher the next trading day.

**Models**
- Random Forest
- Logistic Regression
- XGBoost

**Evaluation**
- Chronological train/test split
- Accuracy and precision
- Confusion matrices
- Walk-forward backtesting

The predictors use only information available at the end of the current trading day.

In [1]:
# Install packages if needed
# Run this cell once if a package is not installed.

%pip install yfinance xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## 1. Download S&P 500 data

In [3]:
# Download historical S&P 500 data

sp500 = yf.Ticker("^GSPC").history(period="max")

# Keep the period used in the original project
sp500 = sp500.loc["2020-01-01":"2026-08-30"].copy()

# Keep only the columns needed for this project
sp500 = sp500[["Open", "High", "Low", "Close", "Volume"]]

sp500.head()

,Open,High,Low,Close,Volume
Date,,,,,
2020-01-02 00:00:00-05:00,3244.669922,3258.139893,3235.530029,3257.850098,3459930000
2020-01-03 00:00:00-05:00,3226.360107,3246.149902,3222.340088,3234.850098,3484700000
2020-01-06 00:00:00-05:00,3217.550049,3246.840088,3214.639893,3246.280029,3702460000
2020-01-07 00:00:00-05:00,3241.860107,3244.909912,3232.429932,3237.179932,3435910000
2020-01-08 00:00:00-05:00,3238.590088,3267.070068,3236.669922,3253.050049,3726840000


## 2. Create features

All features below are calculated from current or past market information.  
No future price is used as a predictor.

In [4]:
# Technical features

sp500["MA5"] = sp500["Close"].rolling(window=5).mean()
sp500["MA20"] = sp500["Close"].rolling(window=20).mean()

sp500["Daily_Return"] = sp500["Close"].pct_change()

sp500["Momentum_5"] = sp500["Close"].pct_change(5)

sp500["Volatility_10"] = sp500["Daily_Return"].rolling(window=10).std()

## 3. Create the target

In [5]:
# Target:
# 1 = tomorrow's close is higher than today's close
# 0 = tomorrow's close is not higher than today's close

sp500["Tomorrow"] = sp500["Close"].shift(-1)
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

# Remove rows with missing feature values and the final row,
# which has no known "Tomorrow" price.
sp500 = sp500.dropna().iloc[:-1].copy()

## 4. Define predictors and split the data

In [6]:
predictors = [
    "Close",
    "Volume",
    "Open",
    "High",
    "Low",
    "MA5",
    "MA20",
    "Daily_Return",
    "Momentum_5",
    "Volatility_10"
]

# Chronological split: the latest 100 trading days are kept completely unseen.
train = sp500.iloc[:-100].copy()
test = sp500.iloc[-100:].copy()

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Number of predictors:", len(predictors))

Training rows: 1552
Test rows: 100
Number of predictors: 10


In [7]:
# Check that there are no missing predictor values

print("Missing values in train:", train[predictors].isna().sum().sum())
print("Missing values in test:", test[predictors].isna().sum().sum())

Missing values in train: 0
Missing values in test: 0


## 5. Define the models

In [8]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=10,
    random_state=1
)

# Logistic Regression
# Scaling is important because our features have very different numerical scales.
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=1,
    eval_metric="logloss"
)

## 6. Train and evaluate on the most recent 100 trading days

In [9]:
models = {
    "Random Forest": rf_model,
    "Logistic Regression": log_model,
    "XGBoost": xgb_model
}

test_predictions = {}

for name, model in models.items():
    model.fit(train[predictors], train["Target"])
    predictions = model.predict(test[predictors])

    test_predictions[name] = predictions

    accuracy = accuracy_score(test["Target"], predictions)
    precision = precision_score(test["Target"], predictions, zero_division=0)

    print(name)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("-" * 40)

Random Forest
Accuracy: 0.47
Precision: 0.8333
----------------------------------------
Logistic Regression
Accuracy: 0.57
Precision: 0.57
----------------------------------------
XGBoost
Accuracy: 0.48
Precision: 0.8571
----------------------------------------


In [10]:
# Summary table

results = []

for name, predictions in test_predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(test["Target"], predictions),
        "Precision": precision_score(test["Target"], predictions, zero_division=0)
    })

results = pd.DataFrame(results)
results

,Model,Accuracy,Precision
0,Random Forest,0.47,0.833333
1,Logistic Regression,0.57,0.570000
2,XGBoost,0.48,0.857143


## 7. Confusion matrices

In [11]:
for name, predictions in test_predictions.items():
    cm = confusion_matrix(test["Target"], predictions)
    print(name)
    print(cm)
    print("Rows = actual, columns = predicted")
    print("-" * 40)

Random Forest
[[42  1]
 [52  5]]
Rows = actual, columns = predicted
----------------------------------------
Logistic Regression
[[ 0 43]
 [ 0 57]]
Rows = actual, columns = predicted
----------------------------------------
XGBoost
[[42  1]
 [51  6]]
Rows = actual, columns = predicted
----------------------------------------


In [12]:
# Class distribution in the test set

print(test["Target"].value_counts())

Target
1    57
0    43
Name: count, dtype: int64


In [13]:
# How many UP predictions did each model make?

for name, predictions in test_predictions.items():
    print(name, "UP predictions:", int(predictions.sum()))

Random Forest UP predictions: 6
Logistic Regression UP predictions: 100
XGBoost UP predictions: 7


## 8. Walk-forward backtesting

In [14]:
from sklearn.base import clone

def predict_period(train_data, test_data, predictors, model):
    model = clone(model)

    model.fit(train_data[predictors], train_data["Target"])
    predictions = model.predict(test_data[predictors])

    return pd.DataFrame({
        "Target": test_data["Target"],
        "Predictions": predictions
    }, index=test_data.index)


def backtest(data, model, predictors, start=1000, step=250):
    all_predictions = []

    for i in range(start, len(data), step):
        train_data = data.iloc[:i].copy()
        test_data = data.iloc[i:i + step].copy()

        if len(test_data) == 0:
            break

        period_predictions = predict_period(
            train_data,
            test_data,
            predictors,
            model
        )

        all_predictions.append(period_predictions)

    return pd.concat(all_predictions)

In [15]:
# Run walk-forward backtest for all three models

backtest_results = []

for name, model in models.items():
    predictions = backtest(sp500, model, predictors)

    accuracy = accuracy_score(
        predictions["Target"],
        predictions["Predictions"]
    )

    precision = precision_score(
        predictions["Target"],
        predictions["Predictions"],
        zero_division=0
    )

    backtest_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision
    })

backtest_results = pd.DataFrame(backtest_results)
backtest_results

,Model,Accuracy,Precision
0,Random Forest,0.463190,0.581197
1,Logistic Regression,0.498466,0.554688
2,XGBoost,0.489264,0.580357


## 9. XGBoost probabilities

Instead of only predicting UP/DOWN, XGBoost can also provide the estimated probability of each class.

In [16]:
# Train XGBoost again on the training period
xgb_model.fit(train[predictors], train["Target"])

xgb_probabilities = xgb_model.predict_proba(test[predictors])

# First 10 rows: [probability of DOWN, probability of UP]
print(xgb_probabilities[:10])

[[0.38624078 0.6137592 ]
 [0.37996066 0.62003934]
 [0.34860575 0.65139425]
 [0.38609123 0.61390877]
 [0.38528395 0.61471605]
 [0.35446686 0.64553314]
 [0.3802449  0.6197551 ]
 [0.8342345  0.1657655 ]
 [0.83770204 0.162298  ]
 [0.66673946 0.3332605 ]]


In [17]:
# Distribution of XGBoost UP probabilities

up_probabilities = xgb_probabilities[:, 1]

print("Minimum UP probability:", round(up_probabilities.min(), 4))
print("Maximum UP probability:", round(up_probabilities.max(), 4))
print("Average UP probability:", round(up_probabilities.mean(), 4))

Minimum UP probability: 0.1045
Maximum UP probability: 0.6514
Average UP probability: 0.2061


## Notes

- The test set is chronological rather than randomly shuffled.
- `Tomorrow` is the target and is not included in `predictors`.
- Rolling features use current/past data only.
- The final row is removed because its next-day closing price is unknown.
- Model performance should be evaluated across multiple periods before drawing conclusions.